In [4]:
import random
from typing import Tuple, Dict
from collections import deque

Position = Tuple[int, int]  # (row, col)

In [5]:


# =========================================
# Ambiente 2D
# =========================================

def init_env_2d(rows=4, cols=4, dirt_prob=0.4,
                agent_start_pos: Position = (0, 0),
                seed=None):
    """
    Inizializza un ambiente 2D:
    - rows, cols: dimensioni della griglia
    - dirt_prob: probabilità che ogni cella sia sporca
    - agent_start_pos: posizione iniziale (r, c)
    Restituisce: (rows, cols, dirt, agent_pos, performance)
    """
    if seed is not None:
        random.seed(seed)

    dirt: Dict[Position, bool] = {}
    for r in range(rows):
        for c in range(cols):
            dirt[(r, c)] = (random.random() < dirt_prob)

    agent_pos = agent_start_pos
    performance = 0
    return rows, cols, dirt, agent_pos, performance


def percept_2d(agent_pos: Position, dirt: Dict[Position, bool]):
    """Restituisce ((r, c), è_sporco_qui). Ambiente parzialmente osservabile."""
    return agent_pos, dirt[agent_pos]


def step_env_2d(rows, cols, dirt, agent_pos, performance, action: str):
    """
    Aggiorna ambiente e posizione in base all'azione.
    Azioni: 'LEFT', 'RIGHT', 'UP', 'DOWN', 'SUCK', 'NOOP'
    Restituisce: (dirt, agent_pos, performance).
    """
    performance -= 1  # costo di passo

    r, c = agent_pos

    if action == "SUCK":
        if dirt[agent_pos]:
            dirt[agent_pos] = False
            performance += 10

    elif action == "LEFT":
        if c > 0:
            c -= 1

    elif action == "RIGHT":
        if c < cols - 1:
            c += 1

    elif action == "UP":
        if r > 0:
            r -= 1

    elif action == "DOWN":
        if r < rows - 1:
            r += 1

    elif action == "NOOP":
        pass

    agent_pos = (r, c)
    return dirt, agent_pos, performance


def all_clean_2d(dirt: Dict[Position, bool]) -> bool:
    """True se tutte le celle sono pulite."""
    return not any(dirt.values())


def count_dirty_2d(dirt: Dict[Position, bool]) -> int:
    """Numero di celle sporche rimanenti (utile per debug/lab)."""
    return sum(1 for v in dirt.values() if v)


def show_env_2d(rows, cols, dirt, agent_pos, performance):
    """
    Stampa la griglia con celle a larghezza fissa:
    - " . " per pulito
    - " D " per sporco
    - "(.)" / "(D)" se presente l'agente
    """
    lines = []
    for r in range(rows):
        row_cells = []
        for c in range(cols):
            pos = (r, c)
            dirty_here = dirt[pos]
            if pos == agent_pos:
                ch = "(D)" if dirty_here else "(.)"
            else:
                ch = " D " if dirty_here else " . "
            row_cells.append(ch)
        lines.append("".join(row_cells))
    print("\n".join(lines))
    print(f"(perf={performance}, dirty_left={count_dirty_2d(dirt)})\n")

# =========================================
# Agente 2D model-based (parzialmente osservabile)
# =========================================

def init_agent_2d_model_based(rows, cols, start_pos=(0, 0)):
    """
    Stato dell'agente model-based:
    - rows, cols: dimensioni note della griglia
    - pos: posizione stimata
    - world_belief: mappa interna { (r,c) -> 'unknown'|'dirty'|'clean' }
    """
    world_belief = {
        (r, c): "unknown"
        for r in range(rows)
        for c in range(cols)
    }
    return {
        "rows": rows,
        "cols": cols,
        "pos": start_pos,
        "world_belief": world_belief,
    }


def neighbors_in_bounds(pos, rows, cols):
    """Vicini legali con l'azione corrispondente."""
    r, c = pos
    for dr, dc, action in [(-1, 0, "UP"), (1, 0, "DOWN"),
                           (0, -1, "LEFT"), (0, 1, "RIGHT")]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols:
            yield (nr, nc), action


def bfs_next_step(start, targets, rows, cols):
    """
    Trova il primo passo di un cammino minimo da start
    al più vicino stato in 'targets' (insieme di posizioni).
    Restituisce: azione ('UP','DOWN','LEFT','RIGHT') oppure None.
    """
    if not targets:
        return None

    queue = deque()
    queue.append(start)
    parent = {start: (None, None)}  # pos -> (parent_pos, azione_da_parent_a_pos)

    while queue:
        current = queue.popleft()
        if current in targets:
            # ricostruisci il primo passo
            pos = current
            while parent[pos][0] != start and parent[pos][0] is not None:
                pos = parent[pos][0]
            return parent[pos][1]

        for nxt, action in neighbors_in_bounds(current, rows, cols):
            if nxt not in parent:
                parent[nxt] = (current, action)
                queue.append(nxt)

    return None  # qui non dovrebbe mai arrivare in un grid connesso


def agent_2d_model_based_policy(agent_state, percept):
    """
    Model-based reflex (parzialmente osservabile):
    - mantiene world_belief (mappa interna)
    - se cella creduta sporca -> SUCK
    - altrimenti va verso celle dirty/unknown nella belief.
    """
    rows = agent_state["rows"]
    cols = agent_state["cols"]
    pos = agent_state["pos"]
    belief = agent_state["world_belief"]

    # percetto: solo cella corrente e stato di sporco locale
    (r, c), is_dirty = percept

    # 1. allineiamo la posizione stimata al percetto
    pos = (r, c)

    # 2. aggiorniamo il modello interno con il percetto locale
    if is_dirty:
        belief[pos] = "dirty"
    else:
        belief[pos] = "clean"

    # 3. se qui credo sporco -> SUCK
    if belief[pos] == "dirty":
        action = "SUCK"
    else:
        # 4. scegli un target dalla mappa interna:
        dirty_targets = {p for p, st in belief.items() if st == "dirty"}
        unknown_targets = {p for p, st in belief.items() if st == "unknown"}

        # priorità alle celle sporche, altrimenti esplora unknown
        if dirty_targets:
            next_action = bfs_next_step(pos, dirty_targets, rows, cols)
        else:
            next_action = bfs_next_step(pos, unknown_targets, rows, cols)

        # se non ci sono più né dirty né unknown, resta fermo
        action = next_action if next_action is not None else "NOOP"

    # 5. aggiorniamo la posizione stimata in base all'azione (per il prossimo giro)
    r, c = pos
    if action == "UP" and r > 0:
        pos = (r - 1, c)
    elif action == "DOWN" and r < rows - 1:
        pos = (r + 1, c)
    elif action == "LEFT" and c > 0:
        pos = (r, c - 1)
    elif action == "RIGHT" and c < cols - 1:
        pos = (r, c + 1)
    # SUCK/NOOP non cambiano pos

    new_agent_state = {
        "rows": rows,
        "cols": cols,
        "pos": pos,
        "world_belief": belief,
    }
    return action, new_agent_state


# =========================================
# Simulazione episodio 2D
# =========================================

def run_episode_2d_model_based(rows=4, cols=4, dirt_prob=0.4,
                               steps=50, agent_start_pos=(0, 0),
                               seed=None):
    rows, cols, dirt, agent_pos, performance = init_env_2d(
        rows=rows,
        cols=cols,
        dirt_prob=dirt_prob,
        agent_start_pos=agent_start_pos,
        seed=seed
    )
    agent_state = init_agent_2d_model_based(rows, cols, start_pos=agent_pos)

    for t in range(steps):
        print(f"t={t:02d}:")
        show_env_2d(rows, cols, dirt, agent_pos, performance)

        percept = percept_2d(agent_pos, dirt)
        action, agent_state = agent_2d_model_based_policy(agent_state, percept)

        dirt, agent_pos, performance = step_env_2d(
            rows, cols, dirt, agent_pos, performance, action
        )

        if all_clean_2d(dirt):
            print(f"t={t+1:02d} (tutto pulito):")
            show_env_2d(rows, cols, dirt, agent_pos, performance)
            print("Episodio terminato.")
            break


In [8]:
if __name__ == "__main__":
    run_episode_2d_model_based(
        rows=12,
        cols=9,
        dirt_prob=0.1,
        steps=40,
        agent_start_pos=(0, 0),
        seed=123
    )


t=00:
(D) D  .  .  .  D  .  .  . 
 .  .  .  .  D  .  D  .  D 
 .  .  .  D  .  .  D  .  . 
 .  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  .  . 
 .  .  D  .  .  .  .  .  . 
 .  .  D  .  .  .  .  D  . 
 D  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  D  . 
 .  .  .  .  .  .  .  D  . 
 D  .  D  .  D  .  .  .  . 
 .  .  .  .  .  .  .  .  . 
(perf=0, dirty_left=17)

t=01:
(.) D  .  .  .  D  .  .  . 
 .  .  .  .  D  .  D  .  D 
 .  .  .  D  .  .  D  .  . 
 .  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  .  . 
 .  .  D  .  .  .  .  .  . 
 .  .  D  .  .  .  .  D  . 
 D  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  D  . 
 .  .  .  .  .  .  .  D  . 
 D  .  D  .  D  .  .  .  . 
 .  .  .  .  .  .  .  .  . 
(perf=9, dirty_left=16)

t=02:
 .  D  .  .  .  D  .  .  . 
(.) .  .  .  D  .  D  .  D 
 .  .  .  D  .  .  D  .  . 
 .  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  .  . 
 .  .  D  .  .  .  .  .  . 
 .  .  D  .  .  .  .  D  . 
 D  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  D  . 
 .  .  .